# WorshipBand AI — 콘티 학습 파이프라인 (Colab, GPU)

유튜브 찬양 영상 하나를 넣으면:
1. 오디오만 추출 (yt-dlp)
2. 6-스템으로 분리 (Demucs `htdemucs_6s`: drums/bass/other/vocals/**guitar**/piano — GPU 필요)
3. BPM + 곡 구조(인트로/절/후렴/브릿지/**기타 솔로**/엔딩) + 구간별 악기 믹스를 자동 탐지 (librosa)
4. 앱(`worshipband-ai/`)에 바로 끌어다 쓸 수 있는 `.ts` 파일 하나로 결과를 포장해서 다운로드

> **저작권 주의**: 아래 실행 전 반드시 "이 음원의 저작권/사용권을 보유하고 있음"에 체크해야 실행됩니다.
> 교회가 직접 제작했거나, 명시적 허락을 받았거나, CCLI 등 합법적 라이선스가 있는 음원에만 사용하세요.
> 임의의 유튜브 커버 영상을 무단으로 반주 소스로 쓰는 용도가 아닙니다.

**실행 방법**: 상단 메뉴 `런타임` → `런타임 유형 변경` → **GPU**(T4 등) 선택 후, 셀을 위에서부터 순서대로 실행.

## 1. 설치

In [ ]:
%%capture
!pip install -q yt-dlp demucs librosa
# Colab 은 ffmpeg 가 이미 설치되어 있다. torch 도 GPU 런타임이면 CUDA 버전이 이미 있다.

## 2. 곡 정보 입력 (여기만 바꾸면 됨)

In [ ]:
# @markdown ### 유튜브 링크와 제목을 입력하세요
YOUTUBE_URL = "https://www.youtube.com/watch?v=REPLACE_ME"  # @param {type:"string"}
SONG_TITLE = "곡 제목을 입력하세요"  # @param {type:"string"}

# @markdown 이 음원의 저작권/사용권을 보유하고 있음을 확인합니다 (체크 안 하면 아래에서 실행이 멈춥니다)
RIGHTS_CONFIRMED = False  # @param {type:"boolean"}

assert RIGHTS_CONFIRMED, (
    "음원 저작권/사용권 확인 체크 없이는 진행할 수 없습니다. "
    "교회가 직접 제작했거나, 명시적 허락을 받았거나, CCLI 등 합법 라이선스가 있는 음원에만 사용하세요."
)
import re
SONG_ID = "song_" + re.sub(r"[^a-zA-Z0-9]+", "", SONG_TITLE)[:20].lower() or "song_colab"
print(f"OK — {SONG_TITLE} ({YOUTUBE_URL}) 을(를) {SONG_ID} 로 처리합니다.")

## 3. 오디오 추출
`worshipband-ai/backend/pipeline/extract_audio.py` 와 동일한 로직 (yt-dlp).

In [ ]:
import subprocess, os
from pathlib import Path

WORK_DIR = Path(f"/content/work/{SONG_ID}")
WORK_DIR.mkdir(parents=True, exist_ok=True)

def extract_audio(youtube_url: str, out_dir: Path) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    output_template = str(out_dir / "source.%(ext)s")
    subprocess.run(
        ["yt-dlp", "-x", "--audio-format", "wav", "--audio-quality", "0",
         "-o", output_template, youtube_url],
        check=True,
    )
    return out_dir / "source.wav"

source_wav = extract_audio(YOUTUBE_URL, WORK_DIR / "raw")
print("추출 완료:", source_wav, source_wav.stat().st_size, "bytes")

## 4. 6-스템 분리 (Demucs `htdemucs_6s`, GPU)
`drums / bass / other / vocals / guitar / piano` 를 직접 출력하는 모델이라
이 앱의 악기 구성(드럼/베이스/**기타**/피아노 + synth=other)과 그대로 맞아떨어진다.
GPU가 없으면 CPU로 돌아가긴 하지만 몇 배 더 오래 걸린다 — 런타임을 GPU로 바꿨는지 꼭 확인.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())

DEMUCS_STEMS = ["drums", "bass", "other", "vocals", "guitar", "piano"]
INSTRUMENT_TO_DEMUCS_STEM = {
    "drums": "drums", "bass": "bass", "guitar": "guitar", "piano": "piano", "synth": "other",
}

def separate_stems(source_wav: Path, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    subprocess.run(
        ["python3", "-m", "demucs", "-n", "htdemucs_6s", "-d", device,
         "-o", str(out_dir), str(source_wav)],
        check=True,
    )
    demucs_out_dir = out_dir / "htdemucs_6s" / source_wav.stem
    return {stem: demucs_out_dir / f"{stem}.wav" for stem in DEMUCS_STEMS}

stems = separate_stems(source_wav, WORK_DIR / "stems")
for name, path in stems.items():
    print(name, "->", path, path.exists())

## 5. 구조 탐지 (BPM + 인트로/절/후렴/브릿지/기타 솔로/엔딩 + 구간별 악기 믹스)
`backend/pipeline/detect_sections.py` 와 동일한 로직.

In [ ]:
import librosa, numpy as np
from dataclasses import dataclass
from typing import Dict, List, Optional

@dataclass
class DetectedCue:
    section: str
    at_ms: int
    mix: Optional[Dict[str, float]] = None

def _segment_boundaries(y, sr):
    chroma = librosa.feature.chroma_cqt(y=y, sr=sr)
    sim = librosa.segment.recurrence_matrix(chroma, mode="affinity", sym=True)
    window = 16
    novelty = np.zeros(sim.shape[0])
    for i in range(window, sim.shape[0] - window):
        before = sim[i - window:i, i - window:i].mean()
        after = sim[i:i + window, i:i + window].mean()
        novelty[i] = abs(after - before)
    peak_frames = librosa.util.peak_pick(
        novelty, pre_max=8, post_max=8, pre_avg=8, post_avg=8, delta=0.02, wait=32
    )
    boundary_times = librosa.frames_to_time(peak_frames, sr=sr)
    total_sec = len(y) / sr
    boundary_times = np.concatenate([[0.0], boundary_times, [total_sec]])
    boundary_times = np.unique(np.round(boundary_times, 2))
    segments = list(zip(boundary_times[:-1], boundary_times[1:]))
    return segments if segments else [(0.0, total_sec)]

def _rms(y, sr, start, end):
    clip = y[int(start * sr):int(end * sr)]
    return float(np.sqrt(np.mean(clip**2))) if clip.size else 0.0

def _label_segments(segments, loudness, guitar_y, vocals_y, sr):
    labels = [""] * len(segments)
    chorus_idx = int(np.argmax(loudness))
    labels[0] = "INTRO"
    labels[-1] = "ENDING"
    labels[chorus_idx] = "CHORUS"
    guitar_energies = [_rms(guitar_y, sr, s, e) for s, e in segments]
    avg_guitar_energy = float(np.mean(guitar_energies)) if guitar_energies else 0.0
    for i, (s, e) in enumerate(segments):
        if labels[i]:
            continue
        vocal_energy = _rms(vocals_y, sr, s, e)
        guitar_energy = guitar_energies[i]
        is_instrumental = vocal_energy < 0.02
        guitar_prominent = guitar_energy > avg_guitar_energy
        if is_instrumental and guitar_prominent:
            labels[i] = "SOLO"
        else:
            labels[i] = "VERSE" if i < chorus_idx else "BRIDGE"
    return labels

def _compute_segment_mixes(stems, segments, sr):
    loaded = {
        instrument: librosa.load(str(stems[stem_key]), sr=sr, mono=True)[0]
        for instrument, stem_key in INSTRUMENT_TO_DEMUCS_STEM.items()
    }
    segment_energies = {
        instrument: [_rms(y, sr, s, e) for s, e in segments]
        for instrument, y in loaded.items()
    }
    peak_energy = {
        instrument: (max(energies) or 1.0) for instrument, energies in segment_energies.items()
    }
    return [
        {
            instrument: round(min(1.0, segment_energies[instrument][i] / peak_energy[instrument]), 3)
            for instrument in INSTRUMENT_TO_DEMUCS_STEM
        }
        for i in range(len(segments))
    ]

def _dedupe_consecutive(cues):
    result, last = [], None
    for cue in cues:
        if cue.section != last:
            result.append(cue)
            last = cue.section
    return result

def detect_structure(mixed_wav: Path, stems: Dict[str, Path]):
    y, sr = librosa.load(str(mixed_wav), sr=None, mono=True)
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    segments = _segment_boundaries(y, sr)
    loudness = [_rms(y, sr, s, e) for s, e in segments]
    guitar_y, _ = librosa.load(str(stems["guitar"]), sr=sr, mono=True)
    vocals_y, _ = librosa.load(str(stems["vocals"]), sr=sr, mono=True)
    labels = _label_segments(segments, loudness, guitar_y, vocals_y, sr)
    segment_mixes = _compute_segment_mixes(stems, segments, sr)
    cues = [
        DetectedCue(section=label, at_ms=int(round(start * 1000)), mix=mix)
        for (start, _end), label, mix in zip(segments, labels, segment_mixes)
    ]
    return float(tempo), 4, _dedupe_consecutive(cues)

bpm, beats_per_bar, timeline_cues = detect_structure(source_wav, stems)
print(f"BPM: {bpm:.1f}")
for cue in timeline_cues:
    print(f"  {cue.at_ms:>7} ms  {cue.section:<8} mix={cue.mix}")

## 6. mp3 변환 + base64 인코딩
웹으로 테스트할 때 별도 오디오 파일을 네트워크로 다시 받아오는 방식은 호스팅 환경에 따라
재생이 실패하는 걸 실제로 겪었다 (`MEDIA_ELEMENT_ERROR: Format error`, WAV/MP3 모두 동일하게 발생 —
파일 자체 문제가 아니라 호스팅 쪽 문제로 추정). 그래서 앱 쪽 데모 트랙과 똑같이,
오디오를 **data URI로 파일 안에 통째로 넣는** 방식을 그대로 따른다.

In [ ]:
import base64

def wav_to_mp3_base64(wav_path: Path, bitrate=96) -> str:
    mp3_path = wav_path.with_suffix(".mp3")
    subprocess.run(
        ["ffmpeg", "-y", "-i", str(wav_path), "-b:a", f"{bitrate}k", str(mp3_path)],
        check=True, capture_output=True,
    )
    data = mp3_path.read_bytes()
    return "data:audio/mpeg;base64," + base64.b64encode(data).decode("ascii")

INSTRUMENT_ORDER = ["drums", "bass", "guitar", "piano", "synth"]
track_data_uris = {}
for instrument in INSTRUMENT_ORDER:
    demucs_stem = INSTRUMENT_TO_DEMUCS_STEM[instrument]
    track_data_uris[instrument] = wav_to_mp3_base64(stems[demucs_stem])
    print(instrument, "->", len(track_data_uris[instrument]), "base64 chars")

## 7. 앱에 바로 넣을 `.ts` 파일 생성

In [ ]:
import json

const_names = {i: f"{i.upper()}_{SONG_ID.upper()}" for i in INSTRUMENT_ORDER}

ts_lines = []
ts_lines.append("/**")
ts_lines.append(" * Colab 학습 파이프라인이 생성한 곡 (자동 생성 — 직접 수정하지 말 것).")
ts_lines.append(f" * 원본: {YOUTUBE_URL!r} / 제목: {SONG_TITLE!r}")
ts_lines.append(" * src/constants/embeddedAudio.generated.ts 와 같은 이유로 data URI를 그대로 쓴다.")
ts_lines.append(" */")
ts_lines.append("import { SongConfig } from \"@/types\";")
ts_lines.append("")

for instrument in INSTRUMENT_ORDER:
    ts_lines.append(f"export const {const_names[instrument]} =")
    ts_lines.append(f"  \"{track_data_uris[instrument]}\";")
    ts_lines.append("")

def cue_to_ts(cue) -> str:
    mix_json = json.dumps(cue.mix) if cue.mix else "undefined"
    return (
        "    { section: \"%s\", atMs: %d, mix: %s }," % (cue.section, cue.at_ms, mix_json)
    )

song_var = f"COLAB_SONG_{SONG_ID.upper()}"
ts_lines.append(f"export const {song_var}: SongConfig = {{")
ts_lines.append(f"  id: \"{SONG_ID}\",")
ts_lines.append(f"  title: \"{SONG_TITLE}\",")
ts_lines.append(f"  bpm: {round(bpm, 1)},")
ts_lines.append(f"  beatsPerBar: {beats_per_bar},")
ts_lines.append("  baseKey: \"UNKNOWN\", // TODO: 키 추정은 아직 미구현")
ts_lines.append("  mode: \"timeline\",")
ts_lines.append(f"  sourceYoutubeUrl: \"{YOUTUBE_URL}\",")
ts_lines.append("  tracks: {")
for instrument in INSTRUMENT_ORDER:
    ts_lines.append(f"    {instrument}: {{ uri: {const_names[instrument]} }},")
ts_lines.append("  },")
ts_lines.append("  timeline: [")
for cue in timeline_cues:
    ts_lines.append(cue_to_ts(cue))
ts_lines.append("  ],")
ts_lines.append("};")

ts_content = "\n".join(ts_lines)
out_path = Path(f"/content/{SONG_ID}.generated.ts")
out_path.write_text(ts_content)
print(f"작성 완료: {out_path} ({out_path.stat().st_size} bytes)")
print()
print(ts_content[:800] + "\n...(생략)...")

## 8. 다운로드
받은 `.ts` 파일을 그대로 Claude 에게 보내거나, `worshipband-ai/src/constants/`에 넣고
`SetlistScreen`/`App.tsx`에서 이 곡을 골라 쓰게 연결하면 된다.

In [ ]:
from google.colab import files
files.download(str(out_path))